In [ ]:
import pandas as pd
import numpy as np

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

# calculate rmssd, sdnn
def calculate_hrv_metrics(ibi_data):
    valid_ibi = pd.Series(ibi_data)
    diff_nn_intervals = np.diff(valid_ibi)
    squared_diffs = np.square(diff_nn_intervals)
    rmssd = np.sqrt(np.mean(squared_diffs))
    sdnn = np.std(valid_ibi, ddof=1)
    return rmssd, sdnn

# load ibi data
ibi_baseline = pd.read_csv(f'{DATA}/ibi.csv')
ibi_01 = pd.read_csv(f'{DATA}/ibi_01.csv')
ibi_02 = pd.read_csv(f'{DATA}/ibi_02.csv')
ibi_03 = pd.read_csv(f'{DATA}/ibi_03.csv')

# physiological bounds filter
ibi_baseline = ibi_baseline[(ibi_baseline['ibi'] > 300) & (ibi_baseline['ibi'] < 2000)]

# baseline hrv metrics
rmssd_baseline, sdnn_baseline = calculate_hrv_metrics(ibi_baseline['ibi'])

# load psychometric data
psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

psychometric_01['Question Start Time'] = pd.to_datetime(psychometric_01['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Start Time'] = pd.to_datetime(psychometric_02['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Start Time'] = pd.to_datetime(psychometric_03['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_01['Question Answer Time'] = pd.to_datetime(psychometric_01['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Answer Time'] = pd.to_datetime(psychometric_02['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Answer Time'] = pd.to_datetime(psychometric_03['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)

psychometric_01 = psychometric_01.dropna(subset=['Question Start Time'])
psychometric_02 = psychometric_02.dropna(subset=['Question Start Time'])
psychometric_03 = psychometric_03.dropna(subset=['Question Start Time'])

# filter relevant questions
types_count = {
    'HADS': 14,
    'STAI-S': 20,
    'STAI-T': 20,
    'BFI': 10,
    'FQ': 24
}

def filter_correct_questions(df, types_count):
    filtered_df = pd.DataFrame()
    for q_type, count in types_count.items():
        filtered_df = pd.concat([filtered_df, df[df['Type'] == q_type].head(count)])
    return filtered_df

questions_01 = filter_correct_questions(psychometric_01, types_count)
questions_02 = filter_correct_questions(psychometric_02, types_count)
questions_03 = filter_correct_questions(psychometric_03, types_count)

# get ibi data
def get_ibi_data(question, ibi_data):
    ibi_data['datetime'] = pd.to_datetime(ibi_data['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    ibi_data_filtered = ibi_data[
        (ibi_data['datetime'] >= question['Question Start Time']) &
        (ibi_data['datetime'] <= question['Question Answer Time'])
    ]
    return ibi_data_filtered['ibi'].values

# detect significant decreases
def detect_significant_decrease(test_metrics, baseline_metrics):
    rmssd_decrease = test_metrics[0] < baseline_metrics[0]
    sdnn_decrease = test_metrics[1] < baseline_metrics[1]
    return rmssd_decrease, sdnn_decrease

# calculate hrv metrics
def calculate_hrv_metrics_for_questions(questions, ibi_data, baseline_metrics):
    results = []
    for _, question in questions.iterrows():
        ibi_values = get_ibi_data(question, ibi_data)
        # physiological bounds filter
        ibi_values = ibi_values[(ibi_values > 300) & (ibi_values < 2000)]
        if len(ibi_values) > 1:
            rmssd, sdnn = calculate_hrv_metrics(ibi_values)
            significant_decreases = detect_significant_decrease((rmssd, sdnn), baseline_metrics)
            results.append({
                'Type': question['Type'],
                'Start Time': question['Question Start Time'],
                'End Time': question['Question Answer Time'],
                'Score': question['Answer'],
                'RMSSD': round(rmssd, 2),
                'SDNN': round(sdnn, 2),
                'RMSSD Decrease': 'Yes' if significant_decreases[0] else 'No',
                'SDNN Decrease': 'Yes' if significant_decreases[1] else 'No'
            })
    return pd.DataFrame(results)

# calculate hrv metrics
baseline_metrics = (rmssd_baseline, sdnn_baseline)
results_01 = calculate_hrv_metrics_for_questions(questions_01, ibi_01, baseline_metrics)
results_02 = calculate_hrv_metrics_for_questions(questions_02, ibi_02, baseline_metrics)
results_03 = calculate_hrv_metrics_for_questions(questions_03, ibi_03, baseline_metrics)

# combine data
results_01['Test'] = 'Test 01'
results_02['Test'] = 'Test 02'
results_03['Test'] = 'Test 03'

combined_results = pd.concat([results_01, results_02, results_03], ignore_index=True)

# reorder columns
combined_results = combined_results[['Test', 'Type', 'Start Time', 'End Time', 'Score', 'RMSSD', 'SDNN', 'RMSSD Decrease', 'SDNN Decrease']]

# save data
combined_results.to_csv(f'{DATA}/QQHRV.csv', index=False)

combined_results.head()